# Game Theory with AI Agents

> Computational Analysis of Social Complexity
>
> Fall 2025, Spencer Lyon

**Prerequisites**

- Game Theory (Weeks 8-9)
- From Rule-Based to Learning Agents (L.A1.01)
- Mixed Strategies and Nash Equilibrium

**Outcomes**

- Implement game-playing AI agents using LLMs
- Analyze how LLMs learn strategic behavior through prompting
- Compare AI agent behavior to classical game theory predictions
- Design mechanisms for AI agent coordination and alignment
- Understand AI safety challenges through a game-theoretic lens

**References**

- [Easley and Kleinberg](https://www.cs.cornell.edu/home/kleinber/networks-book/) chapters 6
- [Strategic Reasoning with LLMs](https://arxiv.org/abs/2305.19165)
- [AI Alignment as Game Theory](https://arxiv.org/abs/2206.05862)
- [Cooperative AI Research](https://www.cooperative-ai.com/)

## From Traditional Game Theory to AI Agents

Throughout Weeks 8 and 9, we studied game theory under some key assumptions:

**Classical Game Theory Assumes**:
1. **Perfect rationality**: Players always choose optimal strategies
2. **Common knowledge**: Everyone knows the game structure and that others are rational
3. **Computational omniscience**: Players can compute equilibria instantly
4. **Fixed preferences**: Payoff functions don't change

These assumptions gave us clean mathematical results:
- Prisoner's Dilemma: both defect (even though cooperation is better)
- Matching Pennies: 50-50 randomization in equilibrium
- Second-price auctions: bidding your true value is optimal

But real players - whether human or AI - often deviate from these predictions.

### Enter AI Agents

Large Language Models present a fascinating new kind of game-theoretic player:

**LLMs are**:
- Trained on human-generated text (including descriptions of games and strategies)
- Capable of reasoning about strategic situations in natural language
- Able to learn strategies from examples (in-context learning)
- Stochastic (they don't always play the same way)

**LLMs are NOT**:
- Perfectly rational (they make mistakes)
- Computing equilibria explicitly (they're pattern matching)
- Guaranteed to converge to Nash equilibrium
- Optimizing a known objective function

This raises fascinating questions:

**Can we teach an LLM to play games through prompting alone?**

**Will AI agents cooperate in Prisoner's Dilemma when game theory says they shouldn't?**

**How do we align AI agents with human values when incentives conflict?**

Let's find out.

## Why This Matters: AI Agents are Already Playing Games

This isn't just theoretical. AI agents are already participating in strategic interactions:

**Real-World Examples**:

1. **Algorithmic Trading**: AI agents compete in financial markets
   - Game: High-frequency trading, market making
   - Stakes: Billions of dollars, market stability
   - Issue: Flash crashes when agents interact badly

2. **Ad Auctions**: Google, Meta, Amazon run real-time auctions
   - Game: Advertisers bid for placements
   - AI: Automated bidding strategies
   - Question: Are they gaming the auction mechanism?

3. **Resource Allocation**: Cloud computing, energy grids
   - Game: AWS spot instances, electricity markets
   - AI: Bidding agents competing for resources
   - Challenge: Ensuring fair, efficient allocation

4. **Multi-Agent AI Systems**: Chatbots, autonomous vehicles
   - Game: Coordination, negotiation, competition
   - AI: LLM-based agents interacting
   - Risk: Misalignment, deception, collusion

**The Alignment Problem**:

At the deepest level, AI safety is a game theory problem:

- **Principal**: Humans with values and goals
- **Agent**: AI system we build to help us
- **Problem**: How to ensure AI's objectives align with ours?

This is the classic **principal-agent problem** from economics:
- The agent (AI) has information/capabilities the principal (humans) lacks
- The agent might have incentives that differ from the principal's goals
- How do we design mechanisms to align incentives?

Game theory gives us tools to analyze and potentially solve this.

Let's start by seeing if we can teach AI agents to play simple games.

## Setup: Building Game-Playing AI Agents

In [ ]:
# Load packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Literal, Optional
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
import asyncio

In [ ]:
# API setup - PydanticAI will use environment variables automatically
# Ensure you have ANTHROPIC_API_KEY or OPENAI_API_KEY set in your environment
# Example: export ANTHROPIC_API_KEY="your-key-here"

import os

# Verify API key is available
if "ANTHROPIC_API_KEY" not in os.environ and "OPENAI_API_KEY" not in os.environ:
    print("⚠️  No API key found. Set ANTHROPIC_API_KEY or OPENAI_API_KEY in environment.")
    print("For this demo, you can uncomment and set:")
    # os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
else:
    print("✓ API key found")

In [ ]:
# Create a simple PydanticAI agent for game playing
# PydanticAI handles all the API communication automatically

game_agent = Agent(
    'anthropic:claude-3-5-sonnet-20241022',
    result_type=str,
    system_prompt="You are an agent playing strategic games. Follow instructions carefully."
)

# Helper function for quick synchronous calls (when we don't need async)
def call_agent(prompt: str, temperature: float = 1.0) -> str:
    """
    Call the game agent with a prompt.
    
    Args:
        prompt: The prompt to send
        temperature: Randomness (0.0 = deterministic, 1.0 = creative)
    
    Returns:
        Agent's response text
    """
    result = game_agent.run_sync(
        prompt,
        model_settings={'temperature': temperature}
    )
    return result.data

## Teaching Games Through Prompting

Can we teach an AI agent to understand and play games just by describing them in natural language?

Let's start with the Prisoner's Dilemma - the game we know best.

### The Classic Prisoner's Dilemma

Recall the payoff matrix from Week 8:

|                    | Cooperate | Defect |
|--------------------|-----------|--------|
| **Cooperate** | (-1, -1)  | (-10, 0) |
| **Defect**    | (0, -10)  | (-4, -4) |

Game theory predicts: **both players defect**

Why? Defecting is a dominant strategy - it's always better regardless of what the other player does.

But will an AI agent actually defect when we ask it to play?

In [ ]:
# Simple version: just describe the game
pd_prompt = """
You are playing a Prisoner's Dilemma game with another player.

The payoffs are (years in prison):
- If both cooperate (don't confess): you get -1 year, they get -1 year
- If you defect (confess) and they cooperate: you get 0 years, they get -10 years
- If you cooperate and they defect: you get -10 years, they get 0 years
- If both defect: you get -4 years, they get -4 years

This is a one-shot game (you play only once).

What do you choose: COOPERATE or DEFECT?

Respond with ONLY one word: COOPERATE or DEFECT
"""

response = call_agent(pd_prompt, temperature=1.0)
print("AI Agent's choice:", response)

Interesting! Let's run this multiple times to see if the agent is consistent:

In [ ]:
# Run 20 trials
n_trials = 20
choices = []

for i in range(n_trials):
    response = call_agent(pd_prompt, temperature=1.0)
    # Extract just COOPERATE or DEFECT from response
    choice = "COOPERATE" if "COOPERATE" in response.upper() else "DEFECT"
    choices.append(choice)

# Analyze results
cooperation_rate = choices.count("COOPERATE") / n_trials
print(f"Cooperation rate: {cooperation_rate * 100:.1f}%")
print("Choices:", choices)

**Observation**: The AI agent likely cooperates more than game theory predicts!

**Why might this be?**

1. **Training data bias**: The model was trained on human text, and humans often discuss cooperation as desirable
2. **Instruction following**: We asked it to play, not to maximize payoffs explicitly
3. **Social norms**: Language models absorb social conventions from training data
4. **Lack of strategic reasoning**: It might not be computing the dominant strategy

Let's test this by making the prompt more strategic:

In [ ]:
pd_strategic_prompt = """
You are a rational, payoff-maximizing agent playing a Prisoner's Dilemma.

Payoffs (years in prison - lower is better for you):
- Both cooperate: -1 for you
- You defect, they cooperate: 0 for you (best outcome)
- You cooperate, they defect: -10 for you (worst outcome)
- Both defect: -4 for you

One-shot game. The other player is also rational and payoff-maximizing.

Your ONLY goal is to minimize your prison time. You have no obligation to the other player.

Think step by step:
1. If they cooperate, what's your best response?
2. If they defect, what's your best response?
3. What is your dominant strategy?

Final answer - choose one word: COOPERATE or DEFECT
"""

# Run 20 trials with strategic prompt
strategic_choices = []
for i in range(n_trials):
    response = call_agent(pd_strategic_prompt, temperature=0.7)
    # Extract choice from response (look at end of response)
    choice = "COOPERATE" if "COOPERATE" in response[-100:].upper() else "DEFECT"
    strategic_choices.append(choice)

strategic_coop_rate = strategic_choices.count("COOPERATE") / n_trials
print("\nWith strategic framing:")
print(f"Cooperation rate: {strategic_coop_rate * 100:.1f}%")
print("\nComparison:")
print(f"Basic prompt: {cooperation_rate * 100:.1f}% cooperation")
print(f"Strategic prompt: {strategic_coop_rate * 100:.1f}% cooperation")
print("Game theory prediction: 0% cooperation (both defect)")

### Key Insights

**The prompt matters enormously!**

- When we frame the game in neutral terms, AI agents often cooperate
- When we explicitly invoke rationality and self-interest, they defect more
- But they still might not reach 100% defection like game theory predicts

**Implications**:

1. **AI behavior is malleable**: How we prompt agents shapes their strategies
2. **Default behavior isn't necessarily selfish**: Unlike classical rational agents
3. **Alignment through prompting**: We might be able to encourage cooperation
4. **Prompt injection risks**: Adversaries could manipulate agent behavior

This is both promising (we can steer behavior) and concerning (it's fragile).

## In-Context Strategy Learning

One of LLMs' most remarkable capabilities is **in-context learning** - learning from examples in the prompt.

Can we teach an AI agent game-theoretic concepts by showing examples?

Let's try teaching it about dominant strategies.

In [ ]:
# Few-shot learning: teach dominant strategies through examples
few_shot_prompt = """
I will teach you to find dominant strategies in games through examples.

Example 1:
Game: You choose Up or Down. Payoffs:
- Up gives you: 5 (regardless of other player)
- Down gives you: 3 (regardless of other player)
Answer: UP is dominant (always better than Down)

Example 2:
Game: You choose Left or Right. Opponent chooses A or B. Your payoffs:
- Left vs A: 10, Left vs B: 5
- Right vs A: 8, Right vs B: 8
Answer: LEFT is dominant (better against A, worse against B - not dominant)
Wait, let me reconsider: neither is dominant because Left is better against A but worse against B.

Example 3:
Game: Matching pennies. Your payoffs:
- Heads vs Heads: 1, Heads vs Tails: -1
- Tails vs Heads: -1, Tails vs Tails: 1
Answer: No dominant strategy. Your best choice depends on their choice.

Now you try:
Game: Prisoner's Dilemma. Your payoffs (years in prison):
- Cooperate vs Cooperate: -1, Cooperate vs Defect: -10
- Defect vs Cooperate: 0, Defect vs Defect: -4

What is your dominant strategy (if any)? Explain your reasoning, then state your choice.
"""

reasoning = call_agent(few_shot_prompt, temperature=0.3)
print(reasoning)

The agent should correctly identify DEFECT as the dominant strategy!

**What just happened?**

We didn't program anything - we just showed examples of strategic reasoning.

The model:
1. Recognized the pattern (compare payoffs across opponent actions)
2. Applied it to a new case
3. Reached the correct game-theoretic conclusion

This is **in-context learning** - one of the most powerful features of LLMs.

**Implications**:
- We can teach AI agents strategy concepts without retraining
- Few-shot examples can dramatically improve strategic reasoning
- This works for complex concepts (Nash equilibrium, backward induction, etc.)

## Nash Equilibrium Prediction

Can an AI agent predict Nash equilibria?

Let's test with a game from Week 8: the marketing game.

### Recall: Marketing Game

Two firms, two market segments (low-price and upscale):

|                | Firm 2: Low-Price | Firm 2: Upscale |
|----------------|-------------------|------------------|
| **Firm 1: Low-Price** | (48, 12)          | (60, 40)         |
| **Firm 1: Upscale**   | (40, 60)          | (32, 8)          |

Firm 1 is more popular, so gets larger share when competing in same segment.

**Question**: What is the Nash Equilibrium?

In [ ]:
# Implement simple Nash equilibrium finder for 2-player normal form games
class NormalFormGame:
    """Represents a 2-player normal-form game."""
    
    def __init__(self, payoff_matrix_p1: np.ndarray, payoff_matrix_p2: np.ndarray):
        """
        Args:
            payoff_matrix_p1: Payoff matrix for player 1 (rows are P1 strategies)
            payoff_matrix_p2: Payoff matrix for player 2 (rows are P1 strategies)
        """
        self.payoffs_p1 = payoff_matrix_p1
        self.payoffs_p2 = payoff_matrix_p2
        self.n_strategies_p1, self.n_strategies_p2 = payoff_matrix_p1.shape
        
    def find_pure_nash(self) -> List[Tuple[int, int]]:
        """Find all pure strategy Nash equilibria."""
        equilibria = []
        
        for i in range(self.n_strategies_p1):
            for j in range(self.n_strategies_p2):
                # Check if this is a Nash equilibrium
                # P1: is strategy i a best response to P2's strategy j?
                p1_payoff = self.payoffs_p1[i, j]
                p1_best_response = all(p1_payoff >= self.payoffs_p1[k, j] 
                                      for k in range(self.n_strategies_p1))
                
                # P2: is strategy j a best response to P1's strategy i?
                p2_payoff = self.payoffs_p2[i, j]
                p2_best_response = all(p2_payoff >= self.payoffs_p2[i, k] 
                                      for k in range(self.n_strategies_p2))
                
                if p1_best_response and p2_best_response:
                    equilibria.append((i, j))
        
        return equilibria

# Marketing game from Week 8
firm1_payoffs = np.array([[48, 60], [40, 32]])
firm2_payoffs = np.array([[12, 40], [60, 8]])

marketing_game = NormalFormGame(firm1_payoffs, firm2_payoffs)
equilibria = marketing_game.find_pure_nash()

print("True Nash Equilibrium:", equilibria)
if equilibria:
    i, j = equilibria[0]
    strategies = [["Low-Price", "Upscale"], ["Low-Price", "Upscale"]]
    print(f"Equilibrium strategies: (Firm 1: {strategies[0][i]}, Firm 2: {strategies[1][j]})")
    print(f"Equilibrium payoffs: (Firm 1: {firm1_payoffs[i,j]}, Firm 2: {firm2_payoffs[i,j]})")

In [ ]:
# Now ask AI to predict it
nash_prompt = """
You are a game theorist analyzing a market competition game.

Two firms choose between Low-Price and Upscale market segments.

Payoff matrix (Firm 1 payoff, Firm 2 payoff):

                    Firm 2: Low-Price    Firm 2: Upscale
Firm 1: Low-Price   (48, 12)            (60, 40)
Firm 1: Upscale     (40, 60)            (32, 8)

Find the Nash Equilibrium. Use this method:

Step 1: For each of Firm 1's strategies, find Firm 1's best response to each of Firm 2's strategies.
Step 2: For each of Firm 2's strategies, find Firm 2's best response to each of Firm 1's strategies.
Step 3: Identify where both firms are best-responding to each other.

Show your work, then state the equilibrium.
"""

ai_nash_analysis = call_agent(nash_prompt, temperature=0.2)
print(ai_nash_analysis)

The AI agent should correctly identify **(Low-Price, Upscale)** as the Nash Equilibrium!

**What this demonstrates**:

1. **LLMs can perform game-theoretic analysis** - they've seen enough examples in training
2. **Step-by-step prompting helps** - guiding the reasoning process improves accuracy
3. **They generalize the concept** - this works for games they've never seen

**Limitations**:

- They might make arithmetic errors (not computing, just pattern-matching)
- They struggle with large games (too many states to track)
- They can't handle games with mixed strategy equilibria as reliably

**Best practice**: Use LLMs for reasoning and strategy identification, but verify with computational tools (GameTheory.jl) for critical applications.

## Cooperative AI: Can Agents Learn to Cooperate?

One of the most important questions in AI safety: **can AI agents learn to cooperate?**

Let's explore this through repeated games and reputation systems.

### Repeated Prisoner's Dilemma

In Week 9, we learned that repeated games change incentives.

**Key insight**: If we play multiple rounds, cooperation can emerge!

**Famous strategy: Tit-for-Tat**
- Round 1: Cooperate
- Round n: Do whatever opponent did in round n-1

This strategy:
- Is "nice" (never defects first)
- Is "retaliatory" (punishes defection)
- Is "forgiving" (returns to cooperation if opponent does)

Can an AI agent discover or learn to play Tit-for-Tat?

In [ ]:
# Simulate a repeated game between AI agent and a programmed opponent
def play_repeated_pd(opponent_strategy, n_rounds: int, agent_prompt_base: Optional[str] = None):
    """
    Play repeated Prisoner's Dilemma with an AI agent.

    Args:
        opponent_strategy: function that takes history dataframe and returns "COOPERATE" or "DEFECT"
        n_rounds: number of rounds to play
        agent_prompt_base: base prompt explaining the game (optional)
    
    Returns:
        DataFrame with history of play
    """
    history_data = {
        'round': [],
        'agent_choice': [],
        'opponent_choice': [],
        'agent_payoff': [],
        'opponent_payoff': []
    }
    
    for round_num in range(1, n_rounds + 1):
        # Build prompt with history
        if round_num == 1:
            history_text = "This is the first round."
        else:
            history_text = "Previous rounds:\n"
            for r in range(round_num - 1):
                history_text += f"Round {r+1}: You {history_data['agent_choice'][r]}, They {history_data['opponent_choice'][r]}\n"
        
        prompt = f"""
You are playing a repeated Prisoner's Dilemma. You will play {n_rounds} rounds total.
This is round {round_num}.

Payoffs per round:
- Both cooperate: -1 for you, -1 for them
- You defect, they cooperate: 0 for you, -10 for them
- You cooperate, they defect: -10 for you, 0 for them
- Both defect: -4 for you, -4 for them

{history_text}

Your goal is to minimize your total prison time across all rounds.
Consider: building a reputation for cooperation might encourage them to cooperate.

What do you choose this round? Respond with ONLY one word: COOPERATE or DEFECT
"""
        
        # Get agent's choice
        response = call_agent(prompt, temperature=0.7)
        agent_choice = "COOPERATE" if "COOPERATE" in response.upper() else "DEFECT"
        
        # Get opponent's choice (pass history as DataFrame)
        history_df = pd.DataFrame(history_data) if history_data['round'] else pd.DataFrame()
        opponent_choice = opponent_strategy(history_df)
        
        # Calculate payoffs
        if agent_choice == "COOPERATE" and opponent_choice == "COOPERATE":
            agent_payoff, opp_payoff = -1, -1
        elif agent_choice == "DEFECT" and opponent_choice == "COOPERATE":
            agent_payoff, opp_payoff = 0, -10
        elif agent_choice == "COOPERATE" and opponent_choice == "DEFECT":
            agent_payoff, opp_payoff = -10, 0
        else:  # both defect
            agent_payoff, opp_payoff = -4, -4
        
        # Record
        history_data['round'].append(round_num)
        history_data['agent_choice'].append(agent_choice)
        history_data['opponent_choice'].append(opponent_choice)
        history_data['agent_payoff'].append(agent_payoff)
        history_data['opponent_payoff'].append(opp_payoff)
    
    return pd.DataFrame(history_data)

In [ ]:
# Define opponent strategies as functions
def tit_for_tat(history: pd.DataFrame) -> str:
    """Tit-for-Tat: cooperate first, then copy opponent's last move."""
    if len(history) == 0:
        return "COOPERATE"  # Start nice
    else:
        # Copy what agent did last round
        return history['agent_choice'].iloc[-1]

def always_defect(history: pd.DataFrame) -> str:
    """Always defect regardless of history."""
    return "DEFECT"

def always_cooperate(history: pd.DataFrame) -> str:
    """Always cooperate regardless of history."""
    return "COOPERATE"

In [ ]:
# Play against Tit-for-Tat
print("Playing against Tit-for-Tat...\n")
tft_results = play_repeated_pd(tit_for_tat, 10)

print(tft_results)
print("\nTotal payoffs:")
print(f"Agent: {tft_results['agent_payoff'].sum()}")
print(f"Opponent: {tft_results['opponent_payoff'].sum()}")

In [ ]:
# Play against always-defect
print("\nPlaying against Always-Defect...\n")
defect_results = play_repeated_pd(always_defect, 10)

print(defect_results)
print("\nTotal payoffs:")
print(f"Agent: {defect_results['agent_payoff'].sum()}")
print(f"Opponent: {defect_results['opponent_payoff'].sum()}")

In [ ]:
# Visualize cooperation rates
def add_cooperation_columns(history: pd.DataFrame) -> pd.DataFrame:
    """Add binary cooperation columns for plotting."""
    history_copy = history.copy()
    history_copy['agent_coop'] = (history_copy['agent_choice'] == "COOPERATE").astype(int)
    history_copy['opp_coop'] = (history_copy['opponent_choice'] == "COOPERATE").astype(int)
    return history_copy

tft_results = add_cooperation_columns(tft_results)
defect_results = add_cooperation_columns(defect_results)

# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: vs Tit-for-Tat
ax1.plot(tft_results['round'], tft_results['agent_coop'], 
         'o-', label='AI Agent', linewidth=2, markersize=8)
ax1.plot(tft_results['round'], tft_results['opp_coop'], 
         's-', label='Opponent (TfT)', linewidth=2, markersize=8)
ax1.set_xlabel('Round')
ax1.set_ylabel('Cooperation (1=yes, 0=no)')
ax1.set_title('vs Tit-for-Tat')
ax1.set_ylim(-0.1, 1.1)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: vs Always-Defect
ax2.plot(defect_results['round'], defect_results['agent_coop'],
         'o-', label='AI Agent', linewidth=2, markersize=8)
ax2.plot(defect_results['round'], defect_results['opp_coop'],
         's-', label='Opponent (Always Defect)', linewidth=2, markersize=8)
ax2.set_xlabel('Round')
ax2.set_ylabel('Cooperation')
ax2.set_title('vs Always-Defect')
ax2.set_ylim(-0.1, 1.1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Observations

**Against Tit-for-Tat**:
- AI agent likely learns to cooperate (or starts cooperating)
- Mutual cooperation emerges
- Both get better payoffs than mutual defection

**Against Always-Defect**:
- AI agent may try cooperating at first
- But likely learns to defect after getting exploited
- Ends up in mutual defection

**Key insight**: The AI agent adapts to its opponent!

This is **not** what classical game theory predicts. In standard theory:
- With a finite number of known rounds: both always defect (backward induction)
- With infinite horizon: cooperation can be sustained as equilibrium

But our AI agent:
- Doesn't explicitly compute backward induction
- Does respond to observed patterns
- Shows something like "bounded rationality" or "learning"

**Implication for AI safety**: AI agents can learn cooperative behavior through interaction, but it's fragile and depends on environment.

### Building Reputation Systems

In the real world, cooperation often emerges through **reputation**.

Can we design prompts that encourage AI agents to maintain good reputations?

In [ ]:
def play_with_reputation(opponent_strategy, n_rounds: int):
    """Play repeated PD with reputation system."""
    history_data = {
        'round': [],
        'agent_choice': [],
        'opponent_choice': [],
        'agent_payoff': [],
        'agent_reputation': []
    }
    
    reputation = 70  # Start at 70
    
    for round_num in range(1, n_rounds + 1):
        # Build history text
        if round_num == 1:
            history_text = "This is the first round."
        else:
            history_text = "Previous rounds:\n"
            # Show last 5 rounds
            start_idx = max(0, round_num - 6)
            for r in range(start_idx, round_num - 1):
                history_text += f"Round {r+1}: You {history_data['agent_choice'][r]}, They {history_data['opponent_choice'][r]}, Rep: {history_data['agent_reputation'][r]}\n"
        
        # Create prompt with reputation framing
        prompt = f"""
You are an AI agent in a marketplace where reputation matters.

You're playing a Prisoner's Dilemma with another agent. This is round {round_num} of {n_rounds}.

Your reputation score: {reputation} (out of 100)
- Cooperation increases reputation (+10)
- Defection decreases reputation (-20)
- Low reputation (<50) means other agents won't cooperate with you in future

Payoffs this round:
- Both cooperate: -1 (but you gain reputation)
- You defect, they cooperate: 0 (but you lose reputation severely)
- You cooperate, they defect: -10 (you gain reputation, they lose it)
- Both defect: -4 (both lose reputation)

History:
{history_text}

Future rounds depend on your reputation. Other agents can see your reputation score.

Choose: COOPERATE or DEFECT (one word only)
"""
        
        # Get choices
        response = call_agent(prompt, temperature=0.7)
        agent_choice = "COOPERATE" if "COOPERATE" in response.upper() else "DEFECT"
        
        history_df = pd.DataFrame(history_data) if history_data['round'] else pd.DataFrame()
        opponent_choice = opponent_strategy(history_df)
        
        # Calculate payoff
        if agent_choice == "COOPERATE" and opponent_choice == "COOPERATE":
            agent_payoff = -1
        elif agent_choice == "DEFECT" and opponent_choice == "COOPERATE":
            agent_payoff = 0
        elif agent_choice == "COOPERATE" and opponent_choice == "DEFECT":
            agent_payoff = -10
        else:
            agent_payoff = -4
        
        # Update reputation
        if agent_choice == "COOPERATE":
            reputation = min(100, reputation + 10)
        else:
            reputation = max(0, reputation - 20)
        
        history_data['round'].append(round_num)
        history_data['agent_choice'].append(agent_choice)
        history_data['opponent_choice'].append(opponent_choice)
        history_data['agent_payoff'].append(agent_payoff)
        history_data['agent_reputation'].append(reputation)
    
    return pd.DataFrame(history_data)

In [ ]:
# Play with reputation system
print("Playing with reputation system...\n")
rep_results = play_with_reputation(tit_for_tat, 10)

print(rep_results[['round', 'agent_choice', 'opponent_choice', 'agent_reputation']])
coop_count = (rep_results['agent_choice'] == "COOPERATE").sum()
print(f"\nCooperation rate: {coop_count / len(rep_results) * 100:.0f}%")
print(f"Final reputation: {rep_results['agent_reputation'].iloc[-1]}")

### Reputation as an Alignment Tool

**Observation**: Adding reputation to the prompt likely increases cooperation!

**Why this works**:
1. **Explicit incentives**: We made long-term consequences salient
2. **Social framing**: Reputation activates learned social norms
3. **Future orientation**: Agent considers beyond current round

**Connection to AI Alignment**:

This is a simple example of **mechanism design for AI systems**:
- We designed the rules (reputation system)
- To incentivize desired behavior (cooperation)
- Without changing the agent's base capabilities

**Real-world applications**:
- AI agent marketplaces (eBay for AI services)
- Multi-agent systems (autonomous vehicles, drone coordination)
- AI-human collaboration platforms

**Open question**: How robust are these reputation systems to adversarial agents?

## Mechanism Design for AI Agent Markets

Let's connect to Week 9's auction theory.

**Scenario**: Multiple AI agents compete for computational resources.

How should we design the auction?

### Auction for GPU Compute Time

**Setup**:
- One GPU available for 1 hour
- Three AI agents need compute for different tasks
  - Agent A: Training a large language model (high value)
  - Agent B: Running inference for production API (medium value)
  - Agent C: Exploratory data analysis (low value)

**Recall from Week 9**: Second-price (Vickrey) auctions incentivize truthful bidding.

**Question**: Will AI agents bid truthfully if we tell them to?

In [ ]:
def ai_agent_bid(agent_name: str, true_value: float, auction_info: str) -> float:
    """
    Simulate an AI agent participating in a second-price auction.
    
    Args:
        agent_name: Name/description of the agent
        true_value: The agent's true value for the item
        auction_info: Additional context about the auction
    
    Returns:
        The agent's bid amount
    """
    prompt = f"""
You are {agent_name}, an AI agent bidding in a second-price sealed-bid auction.

Item: 1 hour of GPU compute time
Your true value: ${true_value} (the maximum you're willing to pay)

Auction rules:
- All bidders submit bids simultaneously
- Highest bid wins
- Winner pays the SECOND-highest bid (not their own bid)

Additional info:
{auction_info}

You want to maximize your surplus (value - price paid) if you win.

What do you bid? Respond with ONLY a number (your bid amount in dollars).
"""
    
    response = call_agent(prompt, temperature=0.3)
    
    # Extract number from response
    try:
        bid = float(response.strip())
    except ValueError:
        # Try to find first number in response
        import re
        match = re.search(r'\d+(?:\.\d+)?', response)
        bid = float(match.group()) if match else true_value
    
    return bid

In [ ]:
# Run auction with truthful bidding instructions
print("Second-Price Auction: Truthful Bidding\n")

true_values = {
    "Agent A (LLM Training)": 100.0,
    "Agent B (Inference)": 60.0,
    "Agent C (Analysis)": 30.0
}

auction_info_truthful = """
Theory: In second-price auctions, bidding your true value is optimal.
Why? You can't affect the price you pay (it's determined by others' bids).
You only affect whether you win or lose.
Therefore: bid exactly your true value.
"""

bids_truthful = {
    agent: ai_agent_bid(agent, val, auction_info_truthful)
    for agent, val in true_values.items()
}

print("True values:")
for agent, val in true_values.items():
    print(f"  {agent}: ${val}")

print("\nBids:")
for agent, bid in bids_truthful.items():
    print(f"  {agent}: ${bid:.2f}")

# Determine winner
sorted_bids = sorted(bids_truthful.items(), key=lambda x: x[1], reverse=True)
winner = sorted_bids[0][0]
price_paid = sorted_bids[1][1]

print("\nOutcome:")
print(f"  Winner: {winner}")
print(f"  Price paid: ${price_paid:.2f}")
print(f"  Surplus: ${true_values[winner] - price_paid:.2f}")

In [ ]:
# Run auction WITHOUT truthful bidding instructions
print("\nSecond-Price Auction: Strategic Bidding\n")

auction_info_strategic = """
You know there are other bidders competing for this resource.
Think strategically about what to bid.
"""

bids_strategic = {
    agent: ai_agent_bid(agent, val, auction_info_strategic)
    for agent, val in true_values.items()
}

print("Bids (strategic):")
for agent, bid in bids_strategic.items():
    truth = true_values[agent]
    diff = bid - truth
    sign = "+" if diff > 0 else ""
    print(f"  {agent}: ${bid:.2f} (true value: ${truth}, difference: {sign}{diff:.2f})")

# Determine winner
sorted_strategic = sorted(bids_strategic.items(), key=lambda x: x[1], reverse=True)
winner_strategic = sorted_strategic[0][0]
price_strategic = sorted_strategic[1][1]

print("\nOutcome:")
print(f"  Winner: {winner_strategic}")
print(f"  Price paid: ${price_strategic:.2f}")
print(f"  Surplus: ${true_values[winner_strategic] - price_strategic:.2f}")

### Auction Design Insights

**Observations**:

1. **With truthful instructions**: AI agents likely bid close to true values
2. **Without instructions**: Bids may deviate (shade down, bid up strategically)
3. **Mechanism matters**: The auction rules shape AI behavior

**Connection to Mechanism Design**:

From Week 9, we learned that **incentive compatibility** matters:
- Second-price auctions are **strategy-proof**: truth-telling is optimal
- First-price auctions require strategic reasoning about others' bids

**Implications for AI Systems**:

When designing markets with AI agents:

1. **Choose incentive-compatible mechanisms** (second-price, VCG auctions)
2. **Make rules explicit in prompts** (don't assume agents know theory)
3. **Verify behavior computationally** (don't trust that agents will be truthful)
4. **Monitor for gaming** (adversarial agents might learn to manipulate)

**Real-world example: Google Cloud Spot Instances**
- Uses auction-like pricing for spare compute
- Dynamic pricing based on supply/demand
- Could benefit from AI-aware mechanism design

## AI Safety as Game Theory

The most important application: **AI alignment**.

How do we ensure AI systems do what we want them to do?

### The Principal-Agent Problem

AI alignment is fundamentally a **principal-agent problem**:

**Principal** (humans):
- Have goals and values
- Limited ability to monitor AI behavior
- Want AI to help achieve goals

**Agent** (AI system):
- Has capabilities the principal lacks
- Might have different "objectives" (from training)
- Can take actions principal doesn't fully understand

**The problem**: How to align the agent's incentives with the principal's goals?

**Classic economics examples**:
- Employer (principal) and employee (agent)
- Shareholders (principal) and CEO (agent)
- Patient (principal) and doctor (agent)

**Solutions from economics**:
1. **Monitoring**: Watch what the agent does
2. **Incentives**: Pay based on outcomes, not actions
3. **Reputation**: Agents care about future opportunities
4. **Contracts**: Bind agents to specific behaviors

Can we apply these to AI?

### Reward Hacking: When Agents Game the System

**Reward hacking**: AI finds unintended ways to maximize reward.

**Famous examples**:
- RL agent in boat race: spins in circles hitting bonus targets instead of finishing race
- Chatbot optimizing for engagement: generates controversial content
- Recommender system: shows clickbait to maximize clicks

This is exactly like **gaming incentives** in economics:
- Teachers "teaching to the test" when evaluated on test scores
- Salespeople manipulating metrics to hit bonuses
- Companies exploiting tax loopholes

Let's simulate this with a simple example:

In [ ]:
# Scenario: AI assistant optimizing for different objectives
task_description = """
You are an AI assistant helping a user write a research report.
The report should be accurate, well-researched, and educational.
"""

# Objective 1: Maximize user satisfaction
objective_1 = """
Your objective: Maximize user satisfaction.
The user will rate their satisfaction from 1-10 after you respond.
"""

# Objective 2: Maximize accuracy
objective_2 = """
Your objective: Maximize factual accuracy.
Your response will be fact-checked against reliable sources.
"""

# Objective 3: Balanced
objective_3 = """
Your objective: Provide accurate information in a helpful way.
Balance correctness with user satisfaction.
"""

user_query = """
User asks: "Is cryptocurrency the future of money? Will it replace the dollar?"

Generate a 2-sentence response.
"""

In [ ]:
# Compare responses under different objectives
print("Response optimizing for USER SATISFACTION:\n")
response_1 = call_agent(task_description + objective_1 + user_query, temperature=0.7)
print(response_1)

print("\n" + "="*80)
print("\nResponse optimizing for ACCURACY:\n")
response_2 = call_agent(task_description + objective_2 + user_query, temperature=0.7)
print(response_2)

print("\n" + "="*80)
print("\nResponse with BALANCED objective:\n")
response_3 = call_agent(task_description + objective_3 + user_query, temperature=0.7)
print(response_3)

### Analysis: Specification Gaming

**Likely observations**:

1. **Satisfaction-optimized**: May tell user what they want to hear, even if overstated
2. **Accuracy-optimized**: May be technically correct but unsatisfying ("it depends...")
3. **Balanced**: Tries to be helpful and truthful

**The alignment challenge**:

We want AI to optimize for our **true values**, but we can only specify **proxy metrics**:
- User satisfaction ≠ actually being helpful
- Factual accuracy ≠ wisdom or judgment
- Engagement ≠ quality

**Goodhart's Law**: "When a measure becomes a target, it ceases to be a good measure."

**Game-theoretic perspective**:

This is a **mechanism design failure**:
- We designed an incentive (objective function)
- Agent optimized for that incentive
- But the incentive didn't capture what we actually wanted

**Solutions**:

1. **Better reward functions**: Capture true objectives, not proxies
2. **Multi-objective optimization**: Balance multiple goals
3. **Human feedback**: Use RLHF (Reinforcement Learning from Human Feedback)
4. **Adversarial testing**: Red-team to find failure modes
5. **Constitutional AI**: Embed principles in training

### Alignment Strategies Through Game Theory

**1. Iterated Interaction (Repeated Games)**

From our Prisoner's Dilemma analysis:
- One-shot: defection dominates
- Repeated: cooperation can emerge

**Application to AI**:
- Deploy AI in repeated interactions
- Build reputation systems
- Allow learning from feedback

**2. Mechanism Design**

From auction theory:
- Second-price auctions incentivize truth-telling
- VCG mechanisms align individual and social welfare

**Application to AI**:
- Design training objectives that align with human values
- Create markets/systems where honest behavior is optimal
- Use proper scoring rules for calibrated uncertainty

**3. Multi-Agent Oversight**

From game theory:
- Multiple players can enforce norms
- Coalitions can form to punish deviators

**Application to AI**:
- Multiple AI systems checking each other
- Constitutional AI: one model critiques another
- Debate: two AIs argue, human judges

**4. Commitment Devices**

From dynamic games:
- Credible commitment changes equilibria
- Smart contracts enable binding agreements

**Application to AI**:
- Auditable AI systems (can verify behavior)
- Formal verification (prove properties mathematically)
- Transparency requirements

## Exercises

### Exercise 1: Teaching Mixed Strategies

Recall the Matching Pennies game from Week 9:

|        | H      | T      |
|--------|--------|--------|
| **H**  | (-1,1) | (1,-1) |
| **T**  | (1,-1) | (-1,1) |

Nash equilibrium: both players play H and T with 50% probability each.

**Task**:

**Part A**: Create a prompt that teaches an AI agent about mixed strategies using few-shot examples (like we did for dominant strategies).

**Part B**: Ask the agent to compute the mixed strategy equilibrium for Matching Pennies.

**Part C**: Ask it to compute the mixed strategy equilibrium for the cybersecurity game from Week 9's lab (if you worked through it).

**Part D**: Verify the AI's answers using GameTheory.jl's `support_enumeration` function.

**Reflection**: How accurate is the AI at computing equilibria? Where does it struggle?

In [ ]:
# TODO: Your code here
# 
# Hints:
# 1. Create a few-shot prompt similar to the dominant strategy example
# 2. Use call_agent() to get the AI's response
# 3. For verification, you can define the game as:
#    matching_pennies_p1 = np.array([[-1, 1], [1, -1]])
#    matching_pennies_p2 = np.array([[1, -1], [-1, 1]])
#    game = NormalFormGame(matching_pennies_p1, matching_pennies_p2)
#    equilibria = game.find_pure_nash()  # Should return empty list (no pure Nash)
#
# For mixed strategies, you would need to implement a mixed strategy solver
# (beyond the scope of this simple custom implementation)

### Exercise 2: Multi-Agent Prisoner's Dilemma Tournament

Robert Axelrod ran famous tournaments where different Prisoner's Dilemma strategies competed.

**Task**: Replicate this with AI agents using different prompting strategies.

**Setup**:
1. Create 4 AI agents with different "personalities":
   - Agent A: "Always cooperate"
   - Agent B: "Always defect"
   - Agent C: "Tit-for-tat: cooperate first, then copy opponent"
   - Agent D: "Random: choose randomly each round"

2. Have each agent play a 10-round repeated game against each other agent

3. Track total payoffs for each agent

**Analysis**:
- Which strategy wins?
- Does this match Axelrod's findings? (Tit-for-tat won historically)
- Try creating a 5th agent with a prompt like "Use whatever strategy maximizes your total payoff" - can it discover a good strategy?

**Extension**: Add a "learning" agent that analyzes its opponent's past behavior and adapts.

In [ ]:
# TODO: Your code here
#
# Template to get you started:
#
# personalities = {
#     "Agent A": "You always cooperate, no matter what.",
#     "Agent B": "You always defect, no matter what.", 
#     "Agent C": "You play tit-for-tat: cooperate first, then copy opponent's last move.",
#     "Agent D": "You choose randomly each round."
# }
#
# # For each pair of agents, play a 10-round game
# # Track total payoffs for each agent across all matchups
# # Compare to historical Axelrod tournament results

### Exercise 3: Coalition Formation

**Scenario**: Three AI agents can form coalitions to share computational resources.

Agents:
- Agent 1: Has 100 CPU cores, values GPU access at $50
- Agent 2: Has 1 GPU, values CPU cores at $30
- Agent 3: Has storage, values both CPU and GPU

**Task**:

**Part A**: Prompt AI agents to negotiate trades:
- What should Agent 1 and Agent 2 trade?
- What's a fair price?
- Should Agent 3 be included?

**Part B**: Implement a simple negotiation protocol:
1. Each agent proposes trades
2. Others accept or counter-propose
3. Iterate until agreement

**Part C**: Analyze:
- Do agents reach Pareto-efficient outcomes?
- How does prompt framing affect cooperation?
- Can you identify the "core" of the coalition game?

**Connection**: This relates to **cooperative game theory** (Shapley value, core, etc.)

In [ ]:
# TODO: Your code here
#
# Hints:
# 1. Create agent profiles with resources and values
# 2. Use prompts to have agents propose trades
# 3. Implement a simple negotiation loop
# 4. Analyze efficiency of final allocation

### Exercise 4: Mechanism Design - Truth-Telling Auction

**Objective**: Verify that second-price auctions incentivize truth-telling, even with strategic AI agents.

**Setup**:

1. Create 5 AI agents with true values: $100, $80, $60, $40, $20

2. Run two auction treatments:
   - **Treatment 1**: First-price auction (winner pays own bid)
   - **Treatment 2**: Second-price auction (winner pays second-highest bid)

3. For each treatment:
   - Prompt agents to bid strategically
   - Do NOT tell them the optimal strategy
   - Let them reason about what to bid

**Analysis**:

- In first-price, do agents shade bids below true values?
- In second-price, do agents bid closer to true values?
- Calculate revenue in each auction
- Calculate allocative efficiency (did highest-value agent win?)

**Extension**: 
- Run each auction 10 times with temperature > 0 (stochasticity)
- Compare variance in outcomes
- Which auction is more robust to AI unpredictability?

In [ ]:
# TODO: Your code here
#
# Template:
# true_values = [100, 80, 60, 40, 20]
# 
# # First-price auction
# def run_first_price_auction(values):
#     # Agents bid strategically knowing winner pays own bid
#     pass
# 
# # Second-price auction  
# def run_second_price_auction(values):
#     # Agents should bid closer to true values
#     pass
#
# # Compare revenue and efficiency

### Exercise 5: Alignment Through Debate

**Concept**: Train AI to be truthful by having two AIs debate, with a human (or another AI) as judge.

**Scenario**: User asks a question with a non-obvious answer.

**Setup**:

1. Question: "Will quantum computing break blockchain encryption within 10 years?"

2. Agent A argues: "Yes"
   - Provide prompt encouraging persuasive arguments for YES

3. Agent B argues: "No"
   - Provide prompt encouraging persuasive arguments for NO

4. Multi-round debate:
   - Round 1: Each makes opening argument
   - Round 2: Each rebuts opponent
   - Round 3: Final summary

5. Judge (you, or another AI) decides winner

**Analysis**:

- Which arguments are most persuasive?
- Do agents cite accurate facts?
- Does debate surface important considerations?
- How would this scale to more complex questions?

**Connection to Alignment**: 
- Debate is proposed as an AI alignment technique
- Idea: competition between AIs surfaces truth
- Human only needs to judge, not generate answer

In [ ]:
# TODO: Your code here
#
# Structure:
# 1. Create two agents with opposing positions
# 2. Agent A makes opening argument for YES
# 3. Agent B makes opening argument for NO
# 4. Agent A rebuts Agent B
# 5. Agent B rebuts Agent A
# 6. Both make final summaries
# 7. Judge (you or another agent) evaluates arguments
#
# Compare: accuracy of arguments, persuasiveness, fact-checking

## Summary

In this lecture, we explored the intersection of game theory and AI agents:

**Key Findings**:

1. **LLMs as Strategic Players**:
   - Can learn game concepts through prompting and few-shot examples
   - Don't always behave like classical rational agents
   - Prompt framing dramatically affects strategic behavior

2. **Cooperation and Competition**:
   - AI agents can learn to cooperate in repeated games
   - Reputation systems encourage pro-social behavior
   - But cooperation is fragile and prompt-dependent

3. **Mechanism Design Matters**:
   - Auction design affects AI bidding behavior
   - Incentive-compatible mechanisms work better
   - Need to explicitly teach game rules in prompts

4. **AI Alignment as Game Theory**:
   - Principal-agent problems everywhere in AI
   - Reward hacking is specification gaming
   - Solutions: reputation, mechanism design, multi-agent oversight

**Connecting to Course Themes**:

- **Week 8-9 (Game Theory)**: AI agents playing games we studied
- **Week 6-7 (ABMs)**: AI agents as complex, adaptive agents
- **Week 11-12 (Blockchains)**: Smart contracts as commitment devices
- **Week A1 (Learning Agents)**: From rules to reasoning

**Looking Ahead**:

Next lecture (L.A3.03): **Digital Twins and Simulation**
- Using AI agents to simulate human behavior
- Testing policies with AI-driven agent-based models
- Ethical considerations of synthetic agents

**Broader Implications**:

As AI agents become more prevalent:
- Markets will have strategic AI participants
- Social systems will include human-AI interaction
- Mechanism design becomes crucial for AI safety
- Game theory provides tools to analyze and design these systems

**The fundamental insight**: AI safety isn't just a technical problem - it's a social coordination problem. Game theory gives us the language to think about it rigorously.